# Simple English RAG — Standalone GPU Ingestion

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/parm2006/SimpleEnglishRag/blob/main/colab_standalone.ipynb)

This 100% self-contained notebook uses Google Colab's free **NVIDIA T4 GPU** to stream, parse, chunk, embed, and ingest the entire Simple English Wikipedia dump (~245,000 chunks) directly into your hosted **Qdrant Cloud** cluster in ~15-20 minutes at **~600–900 chunks/second**.

### Prerequisites:
1. A free [Qdrant Cloud](https://cloud.qdrant.io/) account with an active cluster.
2. Your Qdrant Cloud Cluster URL (e.g. `https://xxx.aws.cloud.qdrant.io`) and API Key.

### Safe Credentials Setup:
- **Option A (Recommended & Secure)**: In the left sidebar, click the **🔑 Secrets** icon, enable notebook access, and add:
  - Name: `QDRANT_URL` → Value: your cluster URL
  - Name: `QDRANT_API_KEY` → Value: your API key
- **Option B**: Paste them directly into the form fields in Cell 2.

### Instructions:
1. Ensure GPU is enabled: **Runtime** > **Change runtime type** > select **T4 GPU**.
2. Provide your Qdrant credentials in Cell 2 (or via Colab Secrets 🔑).
3. Click **Runtime** > **Run all** (or press `Ctrl + F9`)!

In [ ]:
# Cell 0: Hardware & Device Verification
import torch

print("=" * 60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f" ACTIVE HARDWARE: NVIDIA GPU ({gpu_name})")
    print(f" VRAM AVAILABLE:  {gpu_mem:.2f} GB")
    print(f" CUDA VERSION:    {torch.version.cuda}")
    print(" Acceleration status: 100% READY FOR HIGH-SPEED GPU INGESTION")
else:
    print(" WARNING: Running on slow CPU! No GPU detected.")
    print("Please go to: Runtime > Change runtime type > select T4 GPU, then re-run!")
print("=" * 60)

In [ ]:
# Cell 1: Install Required Packages
!pip install -q sentence-transformers qdrant-client requests tqdm

In [ ]:
# Cell 2: Connect & Initialize Qdrant Cloud Collection
import os
from qdrant_client import QdrantClient, models

# 1. Attempt to load from Google Colab Secrets (Left sidebar > 🔑 Secrets)
url_secret = None
key_secret = None
try:
    from google.colab import userdata
    url_secret = userdata.get('QDRANT_URL')
    key_secret = userdata.get('QDRANT_API_KEY')
except Exception:
    pass

# 2. Form Parameters (Fallback if Colab Secrets are not set)
QDRANT_URL = "https://your-cluster-id.cloud.qdrant.io" #@param {type:"string"}
QDRANT_API_KEY = "" #@param {type:"string"}
COLLECTION_NAME = "simple_wiki" #@param {type:"string"}

active_url = (url_secret or QDRANT_URL).strip()
active_key = (key_secret or QDRANT_API_KEY).strip()

if not active_url or "your-cluster-id" in active_url:
    raise ValueError("Please provide your Qdrant Cloud URL! Set it in Colab Secrets (🔑) as QDRANT_URL or paste it into the field above.")

if not active_key:
    raise ValueError("Please provide your Qdrant API Key! Set it in Colab Secrets (🔑) as QDRANT_API_KEY or paste it into the field above.")

print(f" Connecting to Qdrant cluster: {active_url}...")
client = QdrantClient(url=active_url, api_key=active_key)

# Auto-initialize collection with INT8 scalar quantization & on-disk storage if it does not exist
if not client.collection_exists(COLLECTION_NAME):
    print(f"Collection '{COLLECTION_NAME}' does not exist yet. Initializing with INT8 scalar quantization & on-disk storage...")
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=384,
            distance=models.Distance.COSINE,
            on_disk=True,
        ),
        quantization_config=models.ScalarQuantization(
            scalar=models.ScalarQuantizationConfig(
                type=models.ScalarType.INT8,
                quantile=0.99,
                always_ram=True,
            )
        ),
        on_disk_payload=True,
        hnsw_config=models.HnswConfigDiff(on_disk=True),
    )
    print(f" Successfully created collection '{COLLECTION_NAME}'!")

info = client.get_collection(COLLECTION_NAME)
print(f" Successfully connected to Qdrant Cloud!")
print(f"Collection '{COLLECTION_NAME}' currently holds {info.points_count:,} points in status '{info.status}'.")

In [ ]:
# Cell 3: Self-Contained Ingestion & GPU Embedding Engine
import bz2
from concurrent.futures import Future, ThreadPoolExecutor
from dataclasses import dataclass
import html
from itertools import batched
import json
from pathlib import Path
import re
import time
from typing import Iterator, Iterable, Optional
import uuid
import xml.etree.ElementTree as ET

from sentence_transformers import SentenceTransformer
import torch

# 1. Load Embedding Model directly onto NVIDIA CUDA GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading BAAI/bge-small-en-v1.5 onto: {device.upper()} ({torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'})...")
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)

# Quick GPU benchmark test
t0 = time.time()
test_vectors = embed_model.encode(["Benchmark test sentence for GPU evaluation."] * 100, batch_size=100, normalize_embeddings=True, show_progress_bar=False)
test_duration = time.time() - t0
print(f" GPU Benchmark: 100 chunks encoded in {test_duration:.2f}s ({100/test_duration:.1f} chunks/sec on GPU!)\n")

# 2. Data Structures
@dataclass
class Document:
    page_id: str
    title: str
    url: str
    text: str

@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    title: str
    url: str
    chunk_index: int
    text: str

# 3. Wikitext Cleaner
def clean_wikitext(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"<!--.*?-->", "", text, flags=re.DOTALL)
    text = re.sub(r"<ref[^>]*>.*?</ref>", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"<ref[^/>]*/>", "", text, flags=re.IGNORECASE)
    text = re.sub(r"</?[a-zA-Z][^>]*>", "", text)
    text = re.sub(r"\[\[(?:File|Image|Category):[^\]]+\]\]", "", text, flags=re.IGNORECASE)
    for _ in range(4):
        text = re.sub(r"\{\{[^{}]*\}\}", "", text)
    text = re.sub(r"\[\[(?:[^|\]]*\|)?([^\]]+)\]\]", r"\1", text)
    text = re.sub(r"\[https?://[^\s\]]+\s*([^\]]*)\]", r"\1", text)
    text = re.sub(r"https?://[^\s\]]+", "", text)
    text = re.sub(r"={2,6}\s*(.*?)\s*={2,6}", r"\n\1\n", text)
    text = re.sub(r"'{2,5}", "", text)
    text = html.unescape(text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

# 4. Checkpoint Tracking
def load_checkpoint(path: Path) -> dict:
    if path.exists():
        try:
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass
    return {"last_page_id": None, "total_processed": 0}

def save_checkpoint(path: Path, last_page_id: str, total_processed: int) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump({"last_page_id": last_page_id, "total_processed": total_processed}, f)
    import os
    os.replace(tmp, path)

# 5. Line-Streaming Dump Parser (O(1) Memory, ~1,000 pages/sec)
def iter_dump_articles(
    dump_path: Path,
    checkpoint_path: Optional[Path] = None,
    min_length: int = 150,
) -> Iterator[Document]:
    checkpoint = load_checkpoint(checkpoint_path) if checkpoint_path else {}
    last_id = checkpoint.get("last_page_id")
    resuming = last_id is not None
    prev_processed = checkpoint.get("total_processed", 0) if resuming else 0
    
    articles_emitted = 0
    current_page_id = None

    with bz2.open(dump_path, "rt", encoding="utf-8", errors="replace") as stream:
        in_page = False
        page_lines = []

        for line in stream:
            if "<page>" in line:
                in_page = True
                page_lines = [line]
            elif in_page:
                page_lines.append(line)
                if "</page>" in line:
                    in_page = False
                    page_xml = "".join(page_lines)
                    page_lines = []

                    try:
                        root = ET.fromstring(page_xml)
                    except Exception:
                        continue

                    ns = root.findtext("{*}ns") or root.findtext("ns")
                    if ns != "0":
                        continue

                    if root.find("{*}redirect") is not None or root.find("redirect") is not None:
                        continue

                    page_id = root.findtext("{*}id") or root.findtext("id") or ""
                    title = root.findtext("{*}title") or root.findtext("title") or ""
                    current_page_id = page_id

                    if resuming:
                        if page_id == last_id:
                            resuming = False
                            print(f" Resumed past article ID {last_id}. Continuing stream...")
                        continue

                    text_elem = root.find(".//{*}text") or root.find(".//text")
                    raw_text = text_elem.text if text_elem is not None and text_elem.text else ""
                    clean_text = clean_wikitext(raw_text)

                    if len(clean_text) >= min_length:
                        doc_url = f"https://simple.wikipedia.org/wiki/{title.replace(' ', '_')}"
                        doc = Document(page_id=page_id, title=title, url=doc_url, text=clean_text)
                        articles_emitted += 1
                        yield doc

                        if checkpoint_path and articles_emitted % 50 == 0:
                            save_checkpoint(checkpoint_path, page_id, prev_processed + articles_emitted)

    if checkpoint_path and current_page_id:
        save_checkpoint(checkpoint_path, current_page_id, prev_processed + articles_emitted)

# 6. Sliding-Window Chunker
def create_chunks(doc: Document, chunk_size: int = 1200, overlap: int = 200) -> list[Chunk]:
    text = doc.text.strip()
    if len(text) < 150:
        return []
    chunks = []
    start = 0
    count = 0
    while start < len(text):
        end = start + chunk_size
        if end < len(text):
            space = text.rfind(" ", start + (chunk_size // 2), end)
            if space != -1:
                end = space
        chunk_text = text[start:end].strip()
        if chunk_text:
            chunks.append(Chunk(
                chunk_id=f"{doc.page_id}-{count}",
                doc_id=doc.page_id,
                title=doc.title,
                url=doc.url,
                chunk_index=count,
                text=chunk_text,
            ))
            count += 1
        start = end - overlap
        if start >= len(text) - overlap:
            break
    return chunks

# 7. GPU Batch Embedding
def chunks_to_points(chunk_batch: list[Chunk]) -> list[models.PointStruct]:
    texts = [c.text for c in chunk_batch]
    vectors = embed_model.encode(texts, batch_size=len(texts), normalize_embeddings=True, show_progress_bar=False)
    points = []
    for c, v in zip(chunk_batch, vectors):
        pt_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, c.chunk_id))
        payload = {
            "chunk_id": c.chunk_id,
            "doc_id": c.doc_id,
            "title": c.title,
            "url": c.url,
            "chunk_index": c.chunk_index,
            "text": c.text,
        }
        points.append(models.PointStruct(
            id=pt_id,
            vector=v.tolist(),
            payload=payload,
        ))
    return points

# 8. Resilient Upsert with Retries
def insert_points_safe(client: QdrantClient, points: list[models.PointStruct], max_retries: int = 5) -> None:
    for attempt in range(1, max_retries + 1):
        try:
            client.upsert(collection_name=COLLECTION_NAME, points=points, wait=True)
            return
        except Exception as e:
            if attempt == max_retries:
                raise
            time.sleep(min(2 ** attempt, 30))

# 9. Pipelined GPU Ingestion Pipeline
def run_ingestion(docs: Iterable[Document], batch_size: int = 256) -> int:
    def chunk_stream():
        for d in docs:
            for c in create_chunks(d):
                yield c

    total_indexed = 0
    t0 = time.time()
    last_future: Optional[Future] = None

    with ThreadPoolExecutor(max_workers=2) as executor:
        for batch in batched(chunk_stream(), batch_size):
            if last_future is not None:
                last_future.result()

            points = chunks_to_points(list(batch))
            total_indexed += len(points)

            last_future = executor.submit(insert_points_safe, client, points)

            rate = total_indexed / max(time.time() - t0, 0.001)
            print(f" Indexed {total_indexed:,} chunks ({rate:.1f} chunks/sec on NVIDIA GPU)...", flush=True)

        if last_future is not None:
            last_future.result()

    return total_indexed

print(" Ingestion engine ready!")

In [ ]:
# Cell 4: Download Dump (~3-5 seconds via wget)
!mkdir -p data
!wget -c --user-agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) SimpleEnglishRAG/0.1" https://dumps.wikimedia.org/simplewiki/latest/simplewiki-latest-pages-articles.xml.bz2 -O data/simplewiki-latest-pages-articles.xml.bz2
dump_file = Path("data/simplewiki-latest-pages-articles.xml.bz2")
print(f"\n Dump ready at: {dump_file} ({dump_file.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Cell 5: Launch GPU Ingestion
checkpoint_file = Path("data/ingest_checkpoint.json")

if checkpoint_file.exists():
    cp = load_checkpoint(checkpoint_file)
    print(f" Resuming from checkpoint: last article ID {cp.get('last_page_id')}, {cp.get('total_processed', 0):,} articles processed.")
else:
    print(" Starting ingestion from the beginning of the Simple English Wikipedia dump.")

print("\n Launching GPU Ingestion into Qdrant Cloud...\n")
articles = iter_dump_articles(dump_file, checkpoint_path=checkpoint_file)
t_start = time.time()
new_chunks = run_ingestion(articles, batch_size=256)
t_end = time.time()

print(f"\n Ingestion Finished! Successfully added {new_chunks:,} chunks in {(t_end-t_start)/60:.1f} minutes.")

In [ ]:
# Cell 6: Verify Final Cloud Collection Stats & Search
final_info = client.get_collection(COLLECTION_NAME)
print(f"\n================ FINAL STATUS ================")
print(f"Collection Name: {COLLECTION_NAME}")
print(f"Total Vectors:   {final_info.points_count:,}")
print(f"Cluster Status:  {final_info.status}")
print(f"==============================================\n")

# Test query
query = "James Webb Space Telescope"
query_vector = embed_model.encode([query], normalize_embeddings=True)[0].tolist()
hits = client.query_points(COLLECTION_NAME, query=query_vector, limit=3).points
print(f"Test Search Results for: '{query}'\n")
for h in hits:
    p = h.payload
    print(f"- [{h.score:.4f}] {p.get('title')}: {p.get('text')[:120]}...")